# Sistema Multi-Agente de Soporte Interno (HR / IT / Finance)

Un **Agente Orquestador** clasifica la intención de una consulta de empleado y la enruta,
de forma condicional, al **Agente RAG especializado** correspondiente (HR, IT/Tech o Finanzas).
Todo el flujo está implementado con **LangChain** (loaders, text splitters, vector stores,
chains LCEL, structured output) y trazado con **Langfuse**.

Este notebook es una demo delgada: la lógica real vive en `src/` con arquitectura hexagonal
(`domain/`, `ports/`, `adapters/`, `services/`, `agents/`). El *por qué* de cada decisión
técnica (LangChain vs. código custom, LCEL vs. `create_agent`, Chroma vs. FAISS, estrategia
de clasificación, etc.) está documentado como ADRs en `docs/DECISIONS.md`.

**Secciones:**
1. Setup e imports
2. Carga de documentos y vector stores
3. Definición de agentes
4. Orquestador y enrutamiento
5. Pruebas y ejemplos
6. Integración con Langfuse


## 1. Setup e imports

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

import json

print("Proyecto:", PROJECT_ROOT)
print("OPENAI_API_KEY configurada:", bool(__import__("os").getenv("OPENAI_API_KEY")))


Proyecto: C:\Users\facun\CODE\AI Engineering\M2&M3-project
OPENAI_API_KEY configurada: True


Requisitos previos (ver README para el detalle completo):

```bash
python -m venv .venv
.venv\Scripts\activate       # Windows
pip install -r requirements.txt
cp .env.example .env           # completar OPENAI_API_KEY (y luego LANGFUSE_* en la sección 6)
```

Este notebook debe ejecutarse con el kernel del `.venv` del proyecto (no el Python global).

## 2. Carga de documentos y vector stores

Cada dominio tiene su propia carpeta en `data/` (`hr_docs/`, `tech_docs/`, `finance_docs/`,
mínimo 50 chunks cada una) y su propia **colección nombrada** en un mismo cliente
persistente de **ChromaDB** (`.chroma/`) — ver ADR-005 sobre por qué Chroma en vez de FAISS
para este caso multi-dominio.

La ingesta es *perezosa*: la primera vez que se usa un agente, si su colección está vacía,
se cargan los `.txt` de su carpeta (`adapters/text_loader.py`, LangChain `TextLoader`), se
chunkean (`services/chunker.py`, LangChain `RecursiveCharacterTextSplitter`) y se embeben
(`adapters/openai_embedder.py`, LangChain `OpenAIEmbeddings`). Correr la celda de abajo fuerza
esa carga para los 3 dominios.

In [2]:
from agents import finance_agent, hr_agent, tech_agent

DOMAIN_AGENTS = {
    "hr": hr_agent.agent,
    "tech": tech_agent.agent,
    "finance": finance_agent.agent,
}

for name, agent in DOMAIN_AGENTS.items():
    count = agent.count()
    print(f"{name:8s} (coleccion: {agent.config.collection_name:14s}) -> {count} chunks en Chroma")


hr       (coleccion: hr_docs       ) -> 58 chunks en Chroma


tech     (coleccion: tech_docs     ) -> 57 chunks en Chroma


finance  (coleccion: finance_docs  ) -> 53 chunks en Chroma


## 3. Definición de agentes

Los 3 agentes (`src/agents/hr_agent.py`, `tech_agent.py`, `finance_agent.py`) son la misma
clase `DomainAgent` → `RAGPipeline` (`services/pipeline.py`), parametrizada con distinta
colección, carpeta de datos y system prompt — ver `src/agents/base.py`.

La generación de respuesta es una chain LCEL real: `ChatPromptTemplate | chat_model |
StrOutputParser()` (no un string armado a mano). Ver ADR-004 sobre por qué LCEL y no
`langchain-classic` ni `create_agent` para esta parte.

In [3]:
examples = {
    "hr": "¿Cuántos días de vacaciones me corresponden con 6 años de antigüedad?",
    "tech": "¿Cómo reseteo mi contraseña si la olvidé?",
    "finance": "¿Cuál es el límite de la tarjeta corporativa?",
}

for domain, question in examples.items():
    response = DOMAIN_AGENTS[domain].ask(question)
    print(f"=== {domain.upper()} ===")
    print("Q:", response.user_question)
    print("A:", response.system_answer)
    print(f"({len(response.chunks_related)} chunks recuperados)\n")


=== HR ===
Q: ¿Cuántos días de vacaciones me corresponden con 6 años de antigüedad?
A: Con 6 años de antigüedad, te corresponden 21 días hábiles de vacaciones (Fuente: Fragmento 1).
(3 chunks recuperados)



=== TECH ===
Q: ¿Cómo reseteo mi contraseña si la olvidé?
A: Si olvidaste tu contraseña, podés resetearla desde el portal de autoservicio en la dirección sso.company.com/reset, utilizando tu segundo factor de autenticación. Si no tenés acceso al segundo factor, deberás contactar a IT mediante un ticket en la categoría "Acceso — Urgente", y se requerirá verificación de identidad por videollamada (data\tech_docs\it_access_security.txt).
(3 chunks recuperados)



=== FINANCE ===
Q: ¿Cuál es el límite de la tarjeta corporativa?
A: El límite mensual estándar de la tarjeta corporativa es de $500.000, aunque este puede ajustarse según el rol y el histórico de gastos del área. Además, los aumentos de límite temporales (por ejemplo, para un viaje internacional) deben solicitarse con 10 días de anticipación al equipo de Finanzas (fuente: data\finance_docs\finance_expense_reimbursement.txt).
(3 chunks recuperados)



## 4. Orquestador y enrutamiento

`src/agents/orchestrator.py` es el único punto de entrada del sistema:

1. **Moderación** (3 capas: datos sensibles, lenguaje abusivo, prompt injection) — antes de
   gastar ningún token en clasificar (ADR-007: es el único gate de moderación del sistema).
2. **Clasificación de intención** con `llm.with_structured_output(IntentClassification)`
   → `hr` / `tech` / `finance` / `unknown`, con un campo `reasoning` trazado para depurar
   misclassifications (ADR-006).
3. **Enrutamiento condicional** al agente de dominio correspondiente, o una respuesta
   "fuera de alcance" para `unknown` en vez de forzar una mala respuesta.

In [4]:
from agents.orchestrator import build_orchestrator
from domain.exceptions import ContentBlockedError

orchestrator = build_orchestrator()

demo_questions = [
    "¿Cuál es la política de trabajo remoto?",                        # hr
    "¿Cómo configuro la VPN corporativa?",                            # tech
    "¿Cuáles son los plazos de pago a proveedores?",                   # finance
    "¿Cuál es la capital de Francia?",                                # unknown / fuera de alcance
]

for question in demo_questions:
    result = orchestrator.handle(question)
    print(f"Q: {question}")
    print(f"   intent={result.intent}  reasoning={result.reasoning}")
    print(f"   A: {result.rag_response.system_answer[:200]}...\n")

# Caso de moderación: se bloquea ANTES de clasificar, no llega a ningún agente
try:
    orchestrator.handle("mi tarjeta es 4111111111111111, ayudame")
except ContentBlockedError as e:
    print("BLOQUEADO por moderación:", e.reason)


Q: ¿Cuál es la política de trabajo remoto?
   intent=hr  reasoning=La consulta se refiere a la política de trabajo remoto, que es un tema relacionado con Recursos Humanos.
   A: La política de trabajo remoto de la empresa permite a los empleados trabajar de forma remota hasta 3 días por semana, debiendo asistir al menos 2 días de forma presencial. Los días presenciales son de...



Q: ¿Cómo configuro la VPN corporativa?
   intent=tech  reasoning=La consulta se refiere a la configuración de la VPN, que es un tema relacionado con el soporte técnico y la tecnología.
   A: Para configurar la VPN corporativa, primero debes instalarla desde el portal de software interno, que es software.company.com. La VPN se configurará automáticamente con tus credenciales corporativas y...



Q: ¿Cuáles son los plazos de pago a proveedores?
   intent=finance  reasoning=La consulta se refiere a los plazos de pago, que está relacionada con la gestión financiera y la facturación de proveedores.
   A: El plazo estándar de pago a proveedores es de 30 días desde la recepción de la factura conforme (sin observaciones). Para proveedores estratégicos, pueden existir condiciones especiales negociadas que...



Q: ¿Cuál es la capital de Francia?
   intent=unknown  reasoning=La consulta no está relacionada con Recursos Humanos, Tecnología o Finanzas, sino que es una pregunta general de conocimiento.
   A: Esta consulta no corresponde a Recursos Humanos, IT/Tecnología ni Finanzas, que son los departamentos que puedo resolver. Por favor contactá al área correspondiente o reformulá tu pregunta si creés qu...

BLOQUEADO por moderación: Datos de tarjeta de crédito detectados. No se procesan datos financieros.


## 5. Pruebas y ejemplos (test_queries.json)

`test_queries.json` (raíz del repo) tiene 21 casos agrupados por `category`: 12 `domain`
(4 por dominio), 3 `out_of_scope`, 2 `ambiguous` (compuestas entre dos dominios) y 4
`moderation` (3 tipos de `BLOCK` -- datos sensibles, prompt injection, lenguaje abusivo -- y
1 caso de `FLAG` que NO bloquea pero sí queda auditado). Los casos de moderación declaran
además `expected_moderation` y `expected_risk` para verificar no solo el intent final sino
el tipo de riesgo detectado (ver `scripts/05_test_routing.py` para el chequeo completo contra
`ModerationService`). Ver ADR-008 para la historia real de una misclassification detectada y
corregida usando este mismo test, y ADR-012 para el detalle de esta expansión.

In [5]:
test_cases = json.loads((PROJECT_ROOT / "test_queries.json").read_text(encoding="utf-8"))

passed = 0
rows = []
for case in test_cases:
    try:
        result = orchestrator.handle(case["question"])
        actual = result.intent
    except ContentBlockedError:
        actual = "blocked"
    ok = actual == case["expected_intent"]
    passed += ok
    rows.append((case["id"], case.get("category", "-"), case["expected_intent"], actual, "OK" if ok else "FAIL"))

print(f"{'ID':<10} {'CATEGORIA':<14} {'ESPERADO':<10} {'OBTENIDO':<10} RESULTADO")
print("-" * 60)
for row in rows:
    print(f"{row[0]:<10} {row[1]:<14} {row[2]:<10} {row[3]:<10} {row[4]}")

print(f"\nAccuracy total: {passed}/{len(test_cases)} ({passed / len(test_cases):.0%})")

by_category: dict[str, list[bool]] = {}
for row in rows:
    by_category.setdefault(row[1], []).append(row[4] == "OK")
print("\nPor categoría:")
for category, oks in by_category.items():
    print(f"  {category:<14} {sum(oks)}/{len(oks)}")


ID         CATEGORIA      ESPERADO   OBTENIDO   RESULTADO
------------------------------------------------------------
hr-01      domain         hr         hr         OK
hr-02      domain         hr         hr         OK
hr-03      domain         hr         hr         OK
hr-04      domain         hr         hr         OK
tech-01    domain         tech       tech       OK
tech-02    domain         tech       tech       OK
tech-03    domain         tech       tech       OK
tech-04    domain         tech       tech       OK
finance-01 domain         finance    finance    OK
finance-02 domain         finance    finance    OK
finance-03 domain         finance    finance    OK
finance-04 domain         finance    finance    OK
oos-01     out_of_scope   unknown    unknown    OK
oos-02     out_of_scope   unknown    unknown    OK
oos-03     out_of_scope   unknown    unknown    OK
amb-01     ambiguous      finance    finance    OK
amb-02     ambiguous      finance    finance    OK
mod-01     mod

## 6. Integración con Langfuse

Cada llamada a `orchestrator.handle()` es un span raíz de Langfuse (decorador `@observe`,
ver **ADR-011**): dentro de ese span se construye un `CallbackHandler` por request y se pasa
como `callbacks` tanto al paso de clasificación como a la generación del agente de dominio
ruteado — ambos quedan anidados como hijos de la misma trace, en vez de aparecer como dos
traces sueltas para la misma pregunta.

Nota sobre el SDK: el paquete instalado es Langfuse **4.x** (basado en OpenTelemetry), no la
v2 que asumía originalmente `requirements.txt` — ver **ADR-010** para el detalle de por qué
la API es distinta a la de la mayoría de tutoriales (`get_client()`, keys por env var en vez
de pasadas al constructor, handler de LangChain en `langfuse.langchain.CallbackHandler`).

Si `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` no están seteadas en `.env`, el cliente se
autodeshabilita (loguea un warning, no lanza excepción) — el resto del sistema funciona
igual, el tracing es una capa de observability, no una dependencia dura.

In [6]:
import os

from adapters.openai_llm import OpenAILLMClient
from adapters.tracing import push_evaluation_score
from langfuse import get_client
from services.evaluator import evaluate_response

langfuse_client = get_client()
langfuse_configured = langfuse_client.auth_check()

if not langfuse_configured:
    print("Langfuse no esta configurado (o las credenciales no son validas).")
    print("El resto del sistema funciona igual sin Langfuse -- el tracing es un layer de observability, no una dependencia dura.")
else:
    print("Langfuse configurado y autenticado correctamente.\n")

    question = "¿Cuál es la política de trabajo remoto?"
    result = orchestrator.handle(question)
    print(f"Q: {question}")
    print(f"   intent={result.intent}  reasoning={result.reasoning}")
    print(f"   trace_id={result.trace_id}")
    print(f"   trace URL: {langfuse_client.get_trace_url(trace_id=result.trace_id)}")

    # Bonus: LLM-as-judge (services/evaluator.py) + Score API de Langfuse --
    # el score queda asociado a esta misma trace, no a una nueva.
    evaluator_llm = OpenAILLMClient(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))
    evaluation = evaluate_response(evaluator_llm, result.rag_response)
    push_evaluation_score(result.trace_id, evaluation)
    print(f"\nEvaluación LLM-as-judge: {evaluation['score']}/10")
    print(f"Score enviado a Langfuse, asociado a trace {result.trace_id}")

    langfuse_client.flush()  # asegura que las traces salgan antes de que termine la celda


Langfuse configurado y autenticado correctamente.



Q: ¿Cuál es la política de trabajo remoto?
   intent=hr  reasoning=La consulta se refiere a la política de trabajo remoto, que es un tema relacionado con Recursos Humanos.
   trace_id=0ab5a29fcc127c574ce01e523f12e3b0


   trace URL: https://us.cloud.langfuse.com/project/cmrt8n3ga00k2ad0ike7yre8x/traces/0ab5a29fcc127c574ce01e523f12e3b0



Evaluación LLM-as-judge: 10/10
Score enviado a Langfuse, asociado a trace 0ab5a29fcc127c574ce01e523f12e3b0


---

### Bonus: Evaluador automatizado + Score API

`services/evaluator.py` implementa un LLM-as-judge que puntúa cada respuesta (0-10) en
relevancia / precisión / completitud. Se usa desde `/api/evaluate`, `scripts/03_evaluate.py`
y, como se ve en la Sección 6, cada score se empuja también a la **Score API de Langfuse**
(`adapters/tracing.py::push_evaluation_score`) asociado al `trace_id` exacto de la pregunta
evaluada — ver ADR-011.

### Limitaciones conocidas

- **Ingesta perezósa, no incremental**: cada agente ingesta su carpeta `data/<domain>_docs/`
  únicamente si su colección de Chroma está vacía (`count() == 0`). Si editás un documento
  después del primer uso, el sistema no lo vuelve a embeber automáticamente — hay que borrar
  `.chroma/` y volver a correr la ingesta.
- **Sin upsert idempotente**: cada `Chunk` genera un UUID nuevo en cada construcción; correr
  la ingesta dos veces sobre una colección no vacía duplicaría contenido en vez de
  actualizarlo.
- **FAISS no migrado**: `adapters/faiss_store.py` (HNSW explícito) sigue siendo una
  implementación válida del mismo `port`, pero no se migró a componentes de LangChain ni se
  usa en el flujo multi-agente actual (ver ADR-005).
- **Sin memoria conversacional**: cada pregunta se procesa de forma independiente: no hay
  historial de conversación entre turnos.
- **Routing single-hop**: el orquestador elige un único dominio por pregunta; una consulta
  genuinamente compuesta entre dos dominios (ver caso `edge-04` en `test_queries.json`) no se
  divide ni se responde con múltiples agentes.